# 工作流模式： 简单胜过复杂

Anthropic 将工作流（预定义路径）从agent（动态工具调用）中区分出来。五种工作流模式覆盖了大多数场景。从直接的API调用出发，只有当步数不能预测的时候才加入agent。

## 问题描述

从简单开始，只有当复杂度打败开销的时候才加。

## 基本概念

### 工作流 vs Agents

- 工作流。  LLMs 和 tools 按照编排好的路径执行。工程师决定图的样子。
- Agent。  LLMs 动态决定tools 以及采取后续步骤。模型决定图的样子。

每种都有用武之地。工作流便宜、快速、容易调试。Agents则解锁了开放式的问题，但让失败模式更难推理。

### 增强LLM

五种模式的基础：用三种能力————搜索（召回）、工具（行动）、记忆（持久化）武装的一个LLM。任何API调用都可以用这些。

### 五种模式

#### 提示词链条

上一个调用的输出是下一个调用的输入。当任务由明确的线性路径组成时使用。在步骤之间可选插入门控。

#### 路由

通过一个LLM分类器选择下游的LLM或者工具。当任务分类与处理模式明显不同时使用。

#### 并行

并发的执行N次LLM调用，然后聚合结果。两种形态：分块（不同的切片），投票（提示词相同，N次调用，多数表决）

#### 编排器-workers

一个编排器LLM动态决定哪些workers（包括LLMs）需要执行并同步结果。与agent的循环类似但是编排器不会无限循环下去。

#### 评估器-优化器

一个LLM提案一个答案，另外一个LLM进行评估。循环迭代直到评估通过。可以看成是Self-Refine的泛化（用了两个LLM而不是一个～）

### 什么时候用工作流

- 可预测的任务。  当你可以枚举所有的步骤时，就应该做。
- 预算紧张的任务。  工作流的步骤是有限制的，但是agent螺旋上上。
- 合规受限任务。  受众需要从图中读取，而不是从轨迹推理。

### 什么时候用Agents

- 开放式研究。  下一步怎么走取决于上一步的结果。
- 变长任务。   几分钟到几小时步数未知的任务。
- 新颖的领域。  你不知道正确的工作流，只能先探索，然后固化。


# 开始编码

对应本章核心：**工作流 vs Agent**、**五种编排模式（链 / 路由 / 并行 / 编排器-workers / 评估器-优化器）**、**可预测图优先于开放循环**。  
先用脚本化玩具跑通五种模式；再用 **PyTorch** 学路由分类；最后用 **LangChain + DeepSeek** 跑评估器-优化器生产循环。


## 1. 教学玩具：Anthropic 五种工作流骨架

- **Chain**：线性步骤 + 可选门控。
- **Route**：分类器 → 分支 handler。
- **Parallel**：分块或投票聚合。
- **Orchestrator-workers**：有限轮次动态派工。
- **Evaluator-optimizer**：提案 ↔ 评估直到通过。


In [ ]:
from __future__ import annotations

import re
from collections import Counter
from dataclasses import dataclass, field
from typing import Any, Callable, Literal

StepFn = Callable[[str, dict[str, Any]], tuple[str, dict[str, Any]]]
RouteFn = Callable[[str, dict[str, Any]], str]
EvalFn = Callable[[str, dict[str, Any]], tuple[bool, str]]


@dataclass
class WorkflowTrace:
    """一次工作流执行的轨迹。"""

    pattern: str
    steps: list[str] = field(default_factory=list)
    output: str = ""


class WorkflowEngine:
    """Anthropic 五种工作流模式的玩具实现（LLM 步骤用脚本函数代替）。"""

    def chain(
        self,
        steps: list[StepFn],
        text: str,
        ctx: dict[str, Any] | None = None,
        gate_after: int | None = None,
        gate: Callable[[str, dict[str, Any]], bool] | None = None,
    ) -> WorkflowTrace:
        """
        Args:
            steps: 线性步骤列表。
            text: 初始输入。
            ctx: 共享上下文。
            gate_after: 在第几步之后做门控。
            gate: 门控函数；返回 False 则提前结束。

        Returns:
            trace: 链条轨迹。
        """
        ctx = dict(ctx or {})
        trace = WorkflowTrace(pattern="chain")
        cur = text
        for i, fn in enumerate(steps):
            cur, ctx = fn(cur, ctx)
            trace.steps.append(f"step{i}:{cur[:60]}")
            if gate_after == i and gate and not gate(cur, ctx):
                trace.output = cur
                trace.steps.append("gate:blocked")
                return trace
        trace.output = cur
        return trace

    def route(
        self,
        classifier: RouteFn,
        branches: dict[str, StepFn],
        text: str,
        ctx: dict[str, Any] | None = None,
    ) -> WorkflowTrace:
        """
        Args:
            classifier: 路由分类器。
            branches: 标签 → handler。
            text: 输入。
            ctx: 上下文。

        Returns:
            trace: 路由轨迹。
        """
        ctx = dict(ctx or {})
        trace = WorkflowTrace(pattern="route")
        label = classifier(text, ctx)
        trace.steps.append(f"route:{label}")
        if label not in branches:
            trace.output = f"[unrouted:{label}]"
            return trace
        out, ctx = branches[label](text, ctx)
        trace.output = out
        trace.steps.append(f"branch:{label}")
        return trace

    def parallel(
        self,
        workers: list[StepFn],
        aggregator: Callable[[list[str], dict[str, Any]], str],
        text: str,
        ctx: dict[str, Any] | None = None,
        mode: Literal["section", "vote"] = "section",
    ) -> WorkflowTrace:
        """
        Args:
            workers: 并行 worker。
            aggregator: 聚合函数。
            text: 输入；``section`` 模式用 ``||`` 分块。
            ctx: 上下文。
            mode: ``section`` 分块 / ``vote`` 投票。

        Returns:
            trace: 并行轨迹。
        """
        ctx = dict(ctx or {})
        trace = WorkflowTrace(pattern=f"parallel:{mode}")
        if mode == "section":
            parts = re.split(r"\|\|", text)
            outs: list[str] = []
            for i, (worker, part) in enumerate(zip(workers, parts)):
                o, ctx = worker(part.strip(), ctx)
                outs.append(o)
                trace.steps.append(f"worker{i}:{o[:40]}")
            trace.output = aggregator(outs, ctx)
            return trace
        outs = []
        for i, worker in enumerate(workers):
            o, ctx = worker(text, ctx)
            outs.append(o)
            trace.steps.append(f"vote{i}:{o[:40]}")
        trace.output = aggregator(outs, ctx)
        return trace

    def orchestrator_workers(
        self,
        orchestrator: Callable[[str, dict[str, Any]], list[str]],
        workers: dict[str, StepFn],
        text: str,
        ctx: dict[str, Any] | None = None,
        max_rounds: int = 3,
    ) -> WorkflowTrace:
        """
        Args:
            orchestrator: 返回本轮 worker 名列表。
            workers: 具名 worker 表。
            text: 任务描述。
            ctx: 上下文。
            max_rounds: 有限轮次（区别于 agent 无限循环）。

        Returns:
            trace: 编排轨迹。
        """
        ctx = dict(ctx or {})
        trace = WorkflowTrace(pattern="orchestrator")
        cur = text
        for r in range(max_rounds):
            plan = orchestrator(cur, ctx)
            trace.steps.append(f"plan{r}:{plan}")
            if not plan:
                break
            parts: list[str] = []
            for name in plan:
                if name not in workers:
                    parts.append(f"[missing:{name}]")
                    continue
                o, ctx = workers[name](cur, ctx)
                parts.append(o)
            cur = " | ".join(parts)
            ctx["round"] = r + 1
            if ctx.get("done"):
                break
        trace.output = cur
        return trace

    def evaluator_optimizer(
        self,
        proposer: StepFn,
        evaluator: EvalFn,
        text: str,
        ctx: dict[str, Any] | None = None,
        max_rounds: int = 4,
    ) -> WorkflowTrace:
        """
        Args:
            proposer: 提案 LLM（脚本函数）。
            evaluator: 评估 LLM（返回 pass + feedback）。
            text: 任务。
            ctx: 上下文。
            max_rounds: 最大迭代轮次。

        Returns:
            trace: 评估-优化轨迹。
        """
        ctx = dict(ctx or {})
        trace = WorkflowTrace(pattern="evaluator_optimizer")
        cur = text
        for r in range(max_rounds):
            cur, ctx = proposer(cur, ctx)
            trace.steps.append(f"propose{r}:{cur[:60]}")
            ok, feedback = evaluator(cur, ctx)
            trace.steps.append(f"eval{r}:{'pass' if ok else feedback}")
            if ok:
                trace.output = cur
                return trace
            ctx["feedback"] = feedback
        trace.output = cur
        return trace


print(
    "WorkflowEngine ready | patterns = chain, route, parallel, orchestrator, evaluator_optimizer"
)


## 2. 玩具示例：五种模式 + 门控 + 投票


In [ ]:
def demo_workflow_patterns() -> None:
    """断言五种模式均可跑通。"""
    eng = WorkflowEngine()

    def extract(t: str, c: dict[str, Any]) -> tuple[str, dict[str, Any]]:
        c["entities"] = re.findall(r"\b[A-Z][a-z]+\b", t)
        return t, c

    def summarize(t: str, c: dict[str, Any]) -> tuple[str, dict[str, Any]]:
        return f"SUM({len(t.split())} words)", c

    def translate(t: str, c: dict[str, Any]) -> tuple[str, dict[str, Any]]:
        return f"ZH:{t}", c

    chain_tr = eng.chain([extract, summarize, translate], "Alice met Bob in Paris")
    assert chain_tr.output.startswith("ZH:SUM")
    print("chain:", chain_tr.output)

    gated = eng.chain(
        [extract, summarize],
        "no names here",
        gate_after=0,
        gate=lambda t, c: len(c.get("entities", [])) > 0,
    )
    assert "gate:blocked" in gated.steps
    print("chain+gate:", gated.steps[-1])

    def cls(t: str, c: dict[str, Any]) -> str:
        return "math" if any(ch.isdigit() for ch in t) else "general"

    route_tr = eng.route(
        cls,
        {
            "math": lambda t, c: (f"MATH:{eval(t)}", c),
            "general": lambda t, c: (f"GEN:{t.upper()}", c),
        },
        "2+3",
    )
    assert route_tr.output == "MATH:5"
    print("route:", route_tr.output)

    sec_tr = eng.parallel(
        [lambda t, c: (f"A:{t.upper()}", c), lambda t, c: (f"B:{len(t)}", c)],
        lambda outs, c: " + ".join(outs),
        "hello||world",
        mode="section",
    )
    assert "A:HELLO" in sec_tr.output and "B:5" in sec_tr.output
    print("parallel section:", sec_tr.output)

    par_tr = eng.parallel(
        [
            lambda t, c: ("positive", c),
            lambda t, c: ("positive", c),
            lambda t, c: ("negative", c),
        ],
        lambda outs, c: Counter(outs).most_common(1)[0][0],
        "ignored",
        mode="vote",
    )
    assert par_tr.output == "positive"
    print("parallel vote:", par_tr.output)

    def orch(_t: str, c: dict[str, Any]) -> list[str]:
        return [] if c.get("round") else ["research", "draft"]

    def research(_t: str, c: dict[str, Any]) -> tuple[str, dict[str, Any]]:
        c["facts"] = ["fact1", "fact2"]
        return "notes ok", c

    def draft(_t: str, c: dict[str, Any]) -> tuple[str, dict[str, Any]]:
        c["done"] = True
        return f"DRAFT using {c.get('facts')}", c

    orch_tr = eng.orchestrator_workers(
        orch,
        {"research": research, "draft": draft},
        "write blog",
    )
    assert "DRAFT" in orch_tr.output
    print("orchestrator:", orch_tr.output)

    attempt = {"n": 0}

    def propose(_t: str, c: dict[str, Any]) -> tuple[str, dict[str, Any]]:
        attempt["n"] += 1
        if attempt["n"] < 2:
            return "bad answer", c
        fb = c.get("feedback", "")
        return f"good answer ({fb})", c

    def evaluate(t: str, c: dict[str, Any]) -> tuple[bool, str]:
        ok = t.startswith("good answer")
        return ok, "need stronger draft" if not ok else "ok"

    eval_tr = eng.evaluator_optimizer(propose, evaluate, "Explain HTN vs AlphaEvolve")
    assert eval_tr.output.startswith("good answer")
    print("evaluator_optimizer:", eval_tr.output)
    print("TOY DEMO OK")


demo_workflow_patterns()


## 3. PyTorch：路由分类头

路由模式的核心是学习「任务类型 → 下游 handler」。用小型 MLP 在 toy 语料上训练三分类（math / code / general）。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

ROUTE_LABELS = ["math", "code", "general"]
ROUTE_VOCAB = ["sum", "add", "python", "def", "hello", "write", "bug", "fix", "number"]


def route_featurize(text: str) -> torch.Tensor:
    """
    Args:
        text: 用户任务。

    Returns:
        x: ``(V,)`` 多热 + 是否含数字。
    """
    toks = set(re.findall(r"[a-z]+", text.lower()))
    x = torch.zeros(len(ROUTE_VOCAB))
    for i, w in enumerate(ROUTE_VOCAB):
        if w in toks:
            x[i] = 1.0
    if any(ch.isdigit() for ch in text):
        x[-1] = 1.0
    return x


class RouteNet(nn.Module):
    """任务路由小网络。"""

    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(len(ROUTE_VOCAB), 16),
            nn.ReLU(),
            nn.Linear(16, len(ROUTE_LABELS)),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def train_route_net(steps: int = 200) -> RouteNet:
    """
    Args:
        steps: 训练步数。

    Returns:
        model: 训练后的路由头。
    """
    samples = [
        ("compute 12 + 8", 0),
        ("sum two numbers", 0),
        ("fix python def bug", 1),
        ("write hello function", 1),
        ("hello write a poem", 2),
        ("explain workflow patterns", 2),
    ]
    X = torch.stack([route_featurize(t) for t, _ in samples])
    y = torch.tensor([lbl for _, lbl in samples])
    model = RouteNet()
    opt = torch.optim.Adam(model.parameters(), lr=0.05)
    for _ in range(steps):
        logits = model(X)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model


def predict_route(model: RouteNet, text: str) -> str:
    """
    Args:
        model: 路由网络。
        text: 任务文本。

    Returns:
        label: math / code / general。
    """
    with torch.no_grad():
        idx = model(route_featurize(text)).argmax().item()
    return ROUTE_LABELS[idx]


def demo_pytorch_router() -> None:
    model = train_route_net()
    queries = ["add numbers 5", "fix python def bug", "write hello poem"]
    print("=== route classifier ===")
    for q in queries:
        print(f"{predict_route(model, q):>8}  <-  {q}")
    assert predict_route(model, "add numbers 5") == "math"
    print("PYTORCH DEMO OK")


demo_pytorch_router()


## 4. 生产级：LangChain 评估器-优化器 + DeepSeek

工具：``classify_workflow`` / ``propose_draft`` / ``evaluate_draft`` / ``finalize_answer``。  
固定**评估器-优化器**图：分类 → 提案 → 评估 →（不通过则修订）→ 定稿。需 ``DEEPSEEK_API_KEY``。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
ROUTE_MODEL = train_route_net(steps=120)

WF_STATE: dict[str, Any] = {
    "task": "",
    "route": "",
    "draft": "",
    "feedback": "",
    "round": 0,
    "passed": False,
    "final": "",
}


class ClassifyArgs(BaseModel):
    """classify_workflow。"""

    task: str = Field(description="User task to classify")


class ProposeArgs(BaseModel):
    """propose_draft。"""

    task: str = Field(description="Task to answer")
    feedback: str = Field(default="", description="Evaluator feedback from last round")


class EvaluateArgs(BaseModel):
    """evaluate_draft。"""

    task: str
    draft: str


class FinalizeArgs(BaseModel):
    """finalize_answer。"""

    draft: str


def reset_workflow_state() -> None:
    """重置生产工作流状态。"""
    global WF_STATE
    WF_STATE = {
        "task": "",
        "route": "",
        "draft": "",
        "feedback": "",
        "round": 0,
        "passed": False,
        "final": "",
    }


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def classify_workflow_impl(task: str) -> str:
    """
    路由模式：PyTorch 分类 + 工作流模式选择。

    Returns:
        json: route + recommended pattern。
    """
    label = predict_route(ROUTE_MODEL, task)
    pattern = {
        "math": "chain",
        "code": "evaluator_optimizer",
        "general": "evaluator_optimizer",
    }[label]
    WF_STATE["task"] = task
    WF_STATE["route"] = label
    return json.dumps({"route": label, "pattern": pattern, "task": task}, ensure_ascii=False)


def propose_draft_impl(task: str, feedback: str = "") -> str:
    """
    提案 LLM：根据任务与反馈写草稿。

    Returns:
        json: draft。
    """
    WF_STATE["round"] += 1
    prompt = (
        f"Task: {task}\n"
        f"Route: {WF_STATE.get('route', 'general')}\n"
        f"Feedback: {feedback or 'none'}\n"
        "Write a concise Chinese answer (<=120 chars). If feedback exists, fix it."
    )
    draft = str(get_llm().invoke(prompt).content).strip()
    WF_STATE["draft"] = draft
    WF_STATE["feedback"] = feedback
    return json.dumps({"draft": draft, "round": WF_STATE["round"]}, ensure_ascii=False)


def evaluate_draft_impl(task: str, draft: str) -> str:
    """
    评估 LLM：结构化 pass/fail（不用 LLM 当 fitness，只做质量门控）。

    Returns:
        json: pass + feedback。
    """
    prompt = (
        f"Task: {task}\nDraft: {draft}\n"
        "Reply ONLY JSON: {\"pass\": true/false, \"feedback\": \"...\"}.\n"
        "Pass if draft is concise, on-topic, and mentions the core idea."
    )
    raw = str(get_llm(temperature=0.0).invoke(prompt).content).strip()
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        data = {"pass": len(draft) >= 20, "feedback": "invalid evaluator json"}
    passed = bool(data.get("pass"))
    feedback = str(data.get("feedback", ""))
    WF_STATE["passed"] = passed
    WF_STATE["feedback"] = feedback
    return json.dumps({"pass": passed, "feedback": feedback}, ensure_ascii=False)


def finalize_answer_impl(draft: str) -> str:
    """
    通过评估后定稿。

    Returns:
        json: final answer。
    """
    if not WF_STATE.get("passed"):
        return json.dumps({"error": "evaluate first"}, ensure_ascii=False)
    WF_STATE["final"] = draft
    return json.dumps({"final": draft, "rounds": WF_STATE["round"]}, ensure_ascii=False)


def build_workflow_tools() -> list[StructuredTool]:
    """
    Returns:
        tools: 评估器-优化器四件套。
    """

    def _classify(**kwargs: Any) -> str:
        return classify_workflow_impl(ClassifyArgs(**kwargs).task)

    def _propose(**kwargs: Any) -> str:
        a = ProposeArgs(**kwargs)
        return propose_draft_impl(a.task, a.feedback)

    def _evaluate(**kwargs: Any) -> str:
        a = EvaluateArgs(**kwargs)
        return evaluate_draft_impl(a.task, a.draft)

    def _finalize(**kwargs: Any) -> str:
        return finalize_answer_impl(FinalizeArgs(**kwargs).draft)

    return [
        StructuredTool.from_function(
            name="classify_workflow",
            description="Route task (math/code/general) and pick workflow pattern.",
            func=_classify,
            args_schema=ClassifyArgs,
        ),
        StructuredTool.from_function(
            name="propose_draft",
            description="Proposer LLM: draft answer; pass feedback to revise.",
            func=_propose,
            args_schema=ProposeArgs,
        ),
        StructuredTool.from_function(
            name="evaluate_draft",
            description="Evaluator LLM: JSON pass/fail + feedback.",
            func=_evaluate,
            args_schema=EvaluateArgs,
        ),
        StructuredTool.from_function(
            name="finalize_answer",
            description="Finalize draft after evaluate_draft returns pass=true.",
            func=_finalize,
            args_schema=FinalizeArgs,
        ),
    ]


WORKFLOW_TOOLS = build_workflow_tools()


def build_workflow_agent() -> Any:
    """
    Returns:
        agent: 固定评估器-优化器图的 LangChain agent。
    """
    system = (
        "You run a fixed evaluator-optimizer workflow (not open-ended ReAct).\n"
        "Steps: classify_workflow -> propose_draft -> evaluate_draft -> "
        "if pass=false then propose_draft with feedback (max 3 revise rounds) -> "
        "finalize_answer when pass=true.\n"
        "Reply in Chinese with the final answer."
    )
    return create_agent(get_llm(), WORKFLOW_TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    """可读轨迹。"""
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args') or {}})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            content = m.content if len(str(m.content)) < 400 else str(m.content)[:400] + "..."
            lines.append(f"OBS[{m.name}]: {content}")
    return "\n".join(lines)


def count_tool_calls(messages: list[BaseMessage]) -> int:
    n = 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            n += len(m.tool_calls)
    return n


def run_workflow_agent(user_text: str) -> dict[str, Any]:
    """
    Args:
        user_text: 用户任务。

    Returns:
        result: agent 结果。
    """
    return build_workflow_agent().invoke({"messages": [HumanMessage(content=user_text)]})


print(f"LangChain Workflow ready | {MODEL}")


## 5. 生产示例：路由 → 提案 → 评估 → 定稿


In [ ]:
def demo_deepseek_workflow() -> None:
    """真实 API；无 key 则 SKIP。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production demo: DEEPSEEK_API_KEY missing")
        return

    reset_workflow_state()
    task = (
        "用 evaluator-optimizer 工作流回答：Anthropic 五种工作流里，"
        "什么时候该用路由而不是开放式 agent？（120 字内中文）"
    )
    result = run_workflow_agent(task)
    print("=== workflow run ===")
    print(format_agent_messages(result["messages"]))
    assert count_tool_calls(result["messages"]) >= 3
    assert WF_STATE.get("final") or WF_STATE.get("passed")
    print("state:", {k: WF_STATE[k] for k in ["route", "round", "passed", "final"]})
    print("PRODUCTION DEMO OK")


demo_deepseek_workflow()
